# Enriching the forecast inputs

> **Historical v1 analysis.** The modelling target in this workflow ends in September 2025; newer supplemental observations remain separate here. For the current 519-game forecast, see [Notebook 20](20_fixed_cohort_2027_forecast.ipynb). [Notebook 17](17_fixed_sources_integration.ipynb) begins the separate v2 workflow.

This step checks a supplemental player-history snapshot and adds calendar features available for both historical and future months. The original category targets stay at September 2025; partial refresh data remains separate so it cannot distort comparisons.

## 1. Validate the additional history

The downloaded snapshot contains 4,994 monthly observations for 55 games through September 2026. All 3,632 overlapping observations agree within 0.011 players. There are 607 observations after the baseline cutoff and 26 app IDs absent from the original catalog. Source metadata and the file checksum are recorded in reports/enrichment/manifest.json.


In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
manifest = json.loads((ROOT / "reports/enrichment/manifest.json").read_text())
pd.DataFrame({"Check": ["Supplemental apps", "Completed rows", "Overlap rows", "Overlap mismatches", "Newer observations", "Apps absent from baseline"],
              "Value": [manifest[key] for key in ["supplemental_apps", "completed_rows", "overlap_rows", "overlap_mismatches", "new_completed_rows", "new_catalog_apps"]]})


Check,Value
Supplemental apps,55
Completed rows,4994
Overlap rows,3632
Overlap mismatches,0
Newer observations,607
Apps absent from baseline,26


## 2. Inspect catalog additions

The game table preserves source labels for review. An external genre label and a successful history download do not replace the project's inclusion rule. These candidate additions require metadata review before entering a revised model cohort.


In [2]:
app_audit = pd.read_csv(ROOT / "reports/enrichment/supplemental_app_audit.csv")
catalog_additions = app_audit.loc[app_audit.eligible.isna(), ["app_id", "name", "app_type", "steam_genres", "first_month", "last_month"]]
catalog_additions


app_id,name,app_type,steam_genres,first_month,last_month
570,Dota 2,game,"Action, Strategy, Free To Play",2012-07-01,2026-09-01
892970,Valheim,game,"Action, Adventure, Indie, RPG",2021-02-01,2026-09-01
1086940,Baldur's Gate 3,game,"Adventure, RPG, Strategy",2020-10-01,2026-09-01
1091500,Cyberpunk 2077,game,RPG,2020-12-01,2026-09-01
1145360,Hades,game,"Action, Indie, RPG",2019-12-01,2026-09-01
1172470,Apex Legends™,game,"Action, Adventure, Free To Play",2020-11-01,2026-09-01
1174180,Red Dead Redemption 2,game,"Action, Adventure",2019-12-01,2026-09-01
1222670,The Sims™ 4,game,"Adventure, Casual, Simulation, Free To Play",2020-06-01,2026-09-01
1245620,ELDEN RING,game,"Action, RPG",2022-03-01,2026-09-01
1364780,Street Fighter™ 6,game,"Action, Adventure",2023-06-01,2026-09-01


## 3. Add features known in advance

Calendar month, days in month and cyclical month features are available through December 2027. Sine and cosine represent the calendar as a circle, so December and January remain adjacent. They give a model an opportunity to learn seasonality; their presence does not establish that a seasonal effect exists.

The 795 original category-month target rows are joined to these fields in data/processed/category_monthly_with_calendar.parquet. Their activity values remain unchanged. Future calendar rows contain no invented player activity.


In [3]:
calendar = pd.read_csv(ROOT / "reports/enrichment/calendar_features.csv")
calendar.tail(12)


month,calendar_month,days_in_month,month_sin,month_cos
2027-01-01,1,31,0.000000e+00,1.000000e+00
2027-02-01,2,28,5.000000e-01,8.660254e-01
2027-03-01,3,31,8.660254e-01,5.000000e-01
2027-04-01,4,30,1.000000e+00,6.123234e-17
2027-05-01,5,31,8.660254e-01,-5.000000e-01
2027-06-01,6,30,5.000000e-01,-8.660254e-01
2027-07-01,7,31,1.224647e-16,-1.000000e+00
2027-08-01,8,31,-5.000000e-01,-8.660254e-01
2027-09-01,9,30,-8.660254e-01,-5.000000e-01
2027-10-01,10,31,-1.000000e+00,-1.836970e-16


## Enrichment findings

The additional history is validated as supplemental evidence, while the modelling table gains reproducible future-known calendar fields. This prepared a historical model comparison on the pinned v1 target series. The later v2 workflow made a separate cohort and cutoff choice rather than silently replacing these targets.

Recorded evidence: [enrichment manifest](../reports/enrichment/manifest.json).
